In [1]:
import numpy as np
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split, ShuffleSplit, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from scipy.stats import mode
from joblib import Parallel, delayed


# --- Step Setup (Prerequisite) ---
# Generate the moons dataset and split it
X, y = make_moons(n_samples=10000, noise=0.4, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


# Find the best hyperparameters from the previous step
param_grid = {'max_leaf_nodes': np.arange(2, 50)}
grid_search_cv = GridSearchCV(DecisionTreeClassifier(random_state=42), param_grid, cv=5)
grid_search_cv.fit(X_train, y_train)
best_params = grid_search_cv.best_params_
print(f"Best hyperparameters found: {best_params}")


# --- Step 8: Grow a Forest (Corrected) ---
# a. Generate 1,000 subsets of the training set
n_trees = 1000
n_instances = 100


# CORRECTED LINE: Replaced 'n_train' with 'train_size'
rs = ShuffleSplit(n_splits=n_trees, train_size=n_instances, random_state=42)


mini_sets = []
for mini_train_index, _ in rs.split(X_train):
    X_mini_train = X_train[mini_train_index]
    y_mini_train = y_train[mini_train_index]
    mini_sets.append((X_mini_train, y_mini_train))


# b. Train one Decision Tree on each subset in parallel
def train_tree(X_train_subset, y_train_subset, params):
    tree = DecisionTreeClassifier(**params, random_state=42)
    tree.fit(X_train_subset, y_train_subset)
    return tree


trees = Parallel(n_jobs=-1)(
    delayed(train_tree)(X_mini, y_mini, best_params) for X_mini, y_mini in mini_sets
)


# Optional: Evaluate the average accuracy of the individual weak trees
individual_accuracies = [accuracy_score(y_test, tree.predict(X_test)) for tree in trees]
print(f"Mean accuracy of individual trees: {np.mean(individual_accuracies):.4f}")


# c. For each test instance, generate predictions from all 1,000 trees
y_preds = np.array([tree.predict(X_test) for tree in trees])


# Use majority vote to get the final prediction for each instance
y_pred_ensemble, _ = mode(y_preds, axis=0, keepdims=False)


# d. Evaluate the ensemble's predictions on the test set
ensemble_accuracy = accuracy_score(y_test, y_pred_ensemble)
print(f"Random Forest ensemble accuracy: {ensemble_accuracy:.4f}")

Best hyperparameters found: {'max_leaf_nodes': np.int64(23)}
Mean accuracy of individual trees: 0.7988
Random Forest ensemble accuracy: 0.8735
